# Figure 3 — signaling histories in the mouse embryo

IRIS is applied to the Pijuan-Sala E6.5–E8.5 gastrulation atlas, a dataset it
never saw in training. Because prediction is per-cell and cell-autonomous, a
signaling history can be read out along a differentiation trajectory.

Validated against known developmental biology:

* BMP activates along the **cardiac** but not the endodermal lineage
* WNT switches **off** before cardiomyocyte differentiation
* RA activates **late** in both (anterior foregut, atrial cardiomyocytes)

Script equivalent: `figures/fig3/fig3_lineage_dynamics.py`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "iris_repro").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "figures"))

import numpy as np
import pandas as pd
from iris_repro import config, data, metrics, plotting, provenance

plt = plotting.set_style()
OUT = config.output_dir("fig3")
print("outputs ->", OUT)

In [ ]:
from fig3.fig3_lineage_dynamics import (LINEAGES, EXPECTED_ORDER,
                                        load_predictions, code_series)

for name, pops in LINEAGES.items():
    print(f"{name:9s} {pops}")

## Load the two lineages

`diffmap_with_labels.csv` carries the diffusion components, the cell-type
annotation and the IRIS combinatorial code per cell. Pseudotime is DC1,
oriented so the primitive streak sits at 0.

In [ ]:
endo = load_predictions("endoderm")
card = load_predictions("cardiac")
endo[["celltype", "code", "stage", "pseudotime"]].head()

In [ ]:
pd.DataFrame({"endoderm": endo["celltype"].value_counts(),
              "cardiac": card["celltype"].value_counts()}).fillna(0).astype(int)

## Pathway activity along pseudotime (Fig. 3d)

In [ ]:
colors = config.palette()

fig, axes = plt.subplots(1, 2, figsize=(5.4, 1.9), sharey=True)
for ax, (name, df) in zip(axes, [("Endodermal", endo), ("Cardiac", card)]):
    pt = df["pseudotime"].to_numpy(float)
    for sig in config.signals():
        plotting.plot_trend(pt, df[f"{sig}_pred"].to_numpy(float), ax=ax,
                            color=colors.get(sig, "k"),
                            label=config.display_name(sig))
    ax.set_title(f"{name} lineage")
    ax.set_xlabel("Diffusion pseudotime")
    ax.set_ylim(0, 1)
axes[0].set_ylabel("Frequency active")
axes[1].legend(frameon=False, ncol=2, fontsize=5)
plt.tight_layout()
plt.show()

Are those trends significant? Two-tailed Spearman, as in the figure legend.

In [ ]:
rows = []
for name, df in [("endoderm", endo), ("cardiac", card)]:
    pt = df["pseudotime"].to_numpy(float)
    for sig in config.signals():
        s = metrics.spearman_trend(pt, df[f"{sig}_pred"].to_numpy(float))
        rows.append({"lineage": name, "signal": config.display_name(sig),
                     "rho": round(s["rho"], 3), "p": s["p"],
                     "direction": "up" if s["rho"] > 0 else "down",
                     "significant": s["p"] < 0.05})
pd.DataFrame(rows)

Read the cardiac rows: BMP and RA rise, WNT falls — the published biology.

## Combination ordering (Fig. 3e/f)

Beyond individual pathways, do whole *combinations* appear in the right order?

In [ ]:
for name, df in [("endoderm", endo), ("cardiac", card)]:
    codes = code_series(df)
    print(f"\n{name}: top combinations")
    print((codes.value_counts().head(6) / len(codes)).round(3).to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(4.0, 3.0))
for ax, (name, df) in zip(axes, [("Endodermal", endo), ("Cardiac", card)]):
    plotting.combination_frequency_plot(code_series(df),
                                        df["pseudotime"].to_numpy(float), ax=ax)
    ax.set_title(f"{name} lineage")
plt.tight_layout()
plt.show()

One-tailed Mann-Whitney on mean pseudotime position: is each combination
significantly later than the one before it?

In [ ]:
rows = []
for name, df in [("endoderm", endo), ("cardiac", card)]:
    codes = code_series(df)
    pt = df["pseudotime"].to_numpy(float)
    expected = [c for c in EXPECTED_ORDER[name] if "?" not in c]
    for a, b in zip(expected, expected[1:]):
        ta, tb = pt[(codes == a).values], pt[(codes == b).values]
        if len(ta) < 3 or len(tb) < 3:
            print(f"  skipping {a} < {b}: n={len(ta)}, {len(tb)}")
            continue
        r = metrics.mannwhitney_greater(tb, ta)
        rows.append({"lineage": name, "comparison": f"{a} before {b}",
                     "n_a": len(ta), "n_b": len(tb), "p": r["p"],
                     "significant": r["p"] < 0.05})
pd.DataFrame(rows)